# Weights + Table 1 and 2

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
IL_CODE = 17

# import all data sets
morg08 = pd.read_csv("../data/csv/morg08.csv")
morg09 = pd.read_csv("../data/csv/morg09.csv")
morg10 = pd.read_csv("../data/csv/morg10.csv")
morg11 = pd.read_csv("../data/csv/morg11.csv")
morg12 = pd.read_csv("../data/csv/morg12.csv")
morg13 = pd.read_csv("../data/csv/morg13.csv")
morg14 = pd.read_csv("../data/csv/morg14.csv")
morg15 = pd.read_csv("../data/csv/morg15.csv")

/var/folders/j8/n0941h8j0wj8kz_20gdz081m0000gn/T/ipykernel_23285/4267059048.py:13: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  morg13 = pd.read_csv("../data/csv/morg13.csv")
/var/folders/j8/n0941h8j0wj8kz_20gdz081m0000gn/T/ipykernel_23285/4267059048.py:15: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  morg15 = pd.read_csv("../data/csv/morg15.csv")


## pooling all years

In [61]:
# Pool all years first (ALL STATES)
pooled = pd.concat([morg08, morg09, morg10, morg11, morg12, morg13, morg14, morg15], ignore_index=True)

# Keep core columns for both Table 1 and Table 2
keep = ["year", "intmonth", "stfips", "class94", "age", "sex", "weight", "earnwt", "earnwke"]
pooled = pooled[keep].copy()

# Coerce types
for c in keep:
    pooled[c] = pd.to_numeric(pooled[c], errors="coerce")

# Keep valid person weights (needed for demographics)
pooled = pooled[(pooled["weight"].notna()) & (pooled["weight"] > 0)].copy()

pooled.head()

,year,intmonth,stfips,class94,age,sex,weight,earnwt,earnwke
0,2008,1,1,4.0,41,1,2846.6161,11113.0751,840.00
1,2008,1,1,5.0,40,2,3118.6074,11730.4212,777.00
2,2008,1,1,4.0,40,1,3719.7807,14536.4886,1634.61
3,2008,1,1,NaN,38,2,2804.8039,11151.4850,NaN
4,2008,1,1,6.0,68,1,3554.8098,14198.8172,NaN


In [66]:
# How is class94 coded?
pooled["class94"].value_counts(dropna=False).sort_index()

class94
1.0      47157
2.0      79425
3.0     122818
4.0    1136611
5.0     114281
6.0      60491
7.0     118469
8.0       1455
NaN     850670
Name: count, dtype: int64

In [67]:
pooled = pooled[pooled["class94"].notna()].copy()
pooled["class94"].value_counts().sort_index()

class94
1.0      47157
2.0      79425
3.0     122818
4.0    1136611
5.0     114281
6.0      60491
7.0     118469
8.0       1455
Name: count, dtype: int64

# Table 1

In [ ]:
# Public definition
is_public = pooled["class94"].isin([1, 2, 3])

is_il = pooled["stfips"] == IL_CODE

pooled["group"] = np.select(
    [
        is_il & is_public,       # Illinois Public
        is_il & (~is_public),    # Illinois Private
        (~is_il) & is_public     # Other Public
    ],
    ["IL Public", "IL Private", "Other Public"],
    default=np.nan
)

# Drop everything not used in Table 1
pooled = pooled[pooled["group"].notna()].copy()

pooled["group"].value_counts()

group
nan             1386013
Other Public     242941
IL Private        45294
IL Public          6459
Name: count, dtype: int64

In [73]:

# Ensure numeric
pooled["earnwke"] = pd.to_numeric(pooled["earnwke"], errors="coerce")

# Drop zero or negative weekly earnings
pooled.loc[pooled["earnwke"] <= 0, "earnwke"] = np.nan

# Log weekly earnings
pooled["log_weekly_earn"] = np.where(
    pooled["earnwke"] > 0,
    np.log(pooled["earnwke"]),
    np.nan)
pooled["earnwke"].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99])


count    1.340375e+06
mean     8.370649e+02
std      6.278205e+02
min      1.000000e-02
1%       5.500000e+01
5%       1.450000e+02
25%      4.000000e+02
50%      6.730000e+02
75%      1.096000e+03
95%      2.211530e+03
99%      2.884610e+03
max      2.884610e+03
Name: earnwke, dtype: float64

In [74]:
# Age indicators
pooled["age_le_30"] = (pooled["age"] <= 30).astype(float)
pooled["age_ge_55"] = (pooled["age"] >= 55).astype(float)

# Female indicator (2 = female in CPS)
pooled["female"] = (pooled["sex"] == 2).astype(float)

In [76]:
#some helper functions
def wmean(x, w):
    x = pd.to_numeric(x, errors="coerce")
    w = pd.to_numeric(w, errors="coerce")
    mask = x.notna() & w.notna() & (w > 0)
    return (x[mask] * w[mask]).sum() / w[mask].sum()

# Wage variables → earnwt
def wage_mean(df, col):
    return wmean(df[col], df["earnwt"])

# Demographic variables → person weight
def demo_mean(df, col):
    return wmean(df[col], df["weight"])

In [77]:
groups = ["IL Public", "IL Private", "Other Public"]

rows = [
    ("Weekly Earnings", "earnwke", "wage"),
    ("Log Weekly Earnings", "log_weekly_earn", "wage"),
    ("Age", "age", "demo"),
    ("Share ≤30", "age_le_30", "demo"),
    ("Share ≥55", "age_ge_55", "demo"),
    ("Female Share", "female", "demo")
]

results = []

for label, col, vtype in rows:
    row = {"Variable": label}
    
    for g in groups:
        dg = pooled[pooled["group"] == g]
        
        if vtype == "wage":
            row[g] = wage_mean(dg, col)
        else:
            row[g] = demo_mean(dg, col)
    
    results.append(row)

table1 = pd.DataFrame(results)
table1

,Variable,IL Public,IL Private,Other Public
0,Weekly Earnings,911.387294,834.738254,932.547649
1,Log Weekly Earnings,6.558811,6.413417,6.602573
2,Age,44.013575,41.084210,44.588065
3,Share ≤30,0.202066,0.289158,0.181096
4,Share ≥55,0.247358,0.196684,0.257825
5,Female Share,0.578426,0.454978,0.569689


In [79]:
pooled["group"].value_counts()

pooled.groupby("group").agg(
    n=("earnwke","size"),
    n_earn=("earnwke", lambda s: s.notna().sum()),
    sum_earnwt=("earnwt", "sum"),
    sum_weight=("weight", "sum")
)

,n,n_earn,sum_earnwt,sum_weight
group,,,,
IL Private,45294,35882,5.724299e+08,1.431116e+08
IL Public,6459,5871,8.020510e+07,2.010191e+07
Other Public,242941,224382,2.070064e+09,5.179221e+08
nan,1386013,1074240,1.263899e+10,3.157203e+09


In [80]:
demo = pooled.groupby("group").apply(lambda d: pd.Series({
    "female_share": wmean(d["female"], d["weight"]),
    "share_le_30": wmean(d["age_le_30"], d["weight"]),
    "share_ge_55": wmean(d["age_ge_55"], d["weight"]),
    "age_mean": wmean(d["age"], d["weight"])
}))
demo

/var/folders/j8/n0941h8j0wj8kz_20gdz081m0000gn/T/ipykernel_7368/1996214185.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  demo = pooled.groupby("group").apply(lambda d: pd.Series({


,female_share,share_le_30,share_ge_55,age_mean
group,,,,
IL Private,0.454978,0.289158,0.196684,41.084210
IL Public,0.578426,0.202066,0.247358,44.013575
Other Public,0.569689,0.181096,0.257825,44.588065
nan,0.454170,0.284798,0.202242,41.175205


# Table 2

In [90]:
#Public sector definition (confirmed earlier)
is_public = pooled["class94"].isin([1,2,3])

#Illinois residents in public sector
il_public = pooled[
    (pooled["stfips"] == IL_CODE) &
    is_public
].copy()

il_public.shape

(6459, 14)

In [83]:
pre = il_public[(il_public["year"] >= 2008) & (il_public["year"] <= 2010)].copy()
post = il_public[(il_public["year"] >= 2012) & (il_public["year"] <= 2014)].copy()

pre.shape, post.shape

((2643, 14), (2306, 14))

In [87]:
#Weekly earnings
pre["weekly_earn"] = pre["earnwke"]
post["weekly_earn"] = post["earnwke"]

#Clean
pre.loc[pre["weekly_earn"] <= 0, "weekly_earn"] = np.nan
post.loc[post["weekly_earn"] <= 0, "weekly_earn"] = np.nan

#Age indicators
pre["age_le_30"] = (pre["age"] <= 30).astype(float)
post["age_le_30"] = (post["age"] <= 30).astype(float)

pre["age_ge_55"] = (pre["age"] >= 55).astype(float)
post["age_ge_55"] = (post["age"] >= 55).astype(float)

#Female
pre["female"] = (pre["sex"] == 2).astype(float)
post["female"] = (post["sex"] == 2).astype(float)

In [88]:

rows = [
    ("Weekly Earnings", "weekly_earn"),
    ("Age", "age"),
    ("Share ≤30", "age_le_30"),
    ("Share ≥55", "age_ge_55"),
    ("Female Share", "female")
]

results = []

for label, col in rows:
    
    pre_vals = pre[col].dropna()
    post_vals = post[col].dropna()
    
    pre_mean = pre_vals.mean()
    post_mean = post_vals.mean()
    diff = post_mean - pre_mean
    
    t_stat, p_val = stats.ttest_ind(post_vals, pre_vals, equal_var=False)
    
    results.append([label, pre_mean, post_mean, diff, p_val])

table2 = pd.DataFrame(
    results,
    columns=["Variable", "Pre-2011", "Post-2011", "Difference", "p-value"]
)

In [89]:
table2

,Variable,Pre-2011,Post-2011,Difference,p-value
0,Weekly Earnings,870.750365,948.190133,77.439768,0.000013
1,Age,44.534241,44.575022,0.040780,0.915845
2,Share ≤30,0.178963,0.186470,0.007507,0.495759
3,Share ≥55,0.239122,0.268864,0.029742,0.016636
4,Female Share,0.583428,0.600173,0.016746,0.231828


# Some verification

In [92]:
#Wage weighted mean
wmean(pooled["earnwke"], pooled["earnwt"])

41.64718743583107

In [93]:
#Demographic weighted mean
wmean(pooled["age"], pooled["weight"])

41.64718743583107

In [95]:
# Unweighted wage mean
unw = pooled["earnwke"].mean()

# Weighted wage mean
wtd = wmean(pooled["earnwke"], pooled["earnwt"])

unw, wtd, wtd / unw

(837.0649046125153, 830.3167055999565, 0.9919382607305911)

In [96]:
pooled.groupby("group")["earnwke"].apply(lambda s: s.notna().mean())

group
IL Private      0.792202
IL Public       0.908964
Other Public    0.923607
nan             0.775058
Name: earnwke, dtype: float64

## Checking Illinois Population

In [104]:
#CHECKING ILLINOIS POPULATION
morg08_IL = morg08[morg08["stfips"] == IL_CODE].copy()
morg08_IL.shape #unweighted

(9672, 99)

In [105]:
morg08_IL["weight"].sum()

29735359.753399998

In [106]:
il_population = morg08_IL["weight"].sum() / 3

il_population

9911786.584466666